# Phase 1 — is the record real?

The formal C1 measurement under the signed thresholds (VIABILITY_PLAN.md §1, estimator as
amended 2026-08-13): grouped split, per-lane train-selected constant, answerable rows,
lane-equal pooling over lanes with ≥100 answerable held-out rows, clustered bootstrap
against Δmin = 0.02. Plus the leakage estimate, decisive-vs-answerable side-by-side, the
rotation readout, the probe gate under the ordering criterion, and the per-archetype
diagnostic. VERDICT.md cites these cells; Codex checkpoint 2 attacks them.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src" / "scripts"))

import numpy as np
import pandas as pd
from feasibility_gate import (
    CLUSTERS_PATH,
    DELTA_MIN,
    fit_and_deltas,
    grouped_split_random,
    pooled_bootstrap,
)

from hybrid_search_rrf_dataset.router import Representation, RouterExperiment, StrategyRouter

MIN_LANE_ROWS = 100  # the amended min-evidence rule

exp = RouterExperiment()
data = exp.load()
cluster_map = pd.read_parquet(CLUSTERS_PATH)
merged = data.merge(cluster_map, on=["dataset", "query_id"], how="left")
clusters = merged["cluster_id"]


def pooled(deltas: pd.DataFrame) -> dict:
    """The amended estimator: lane-equal mean over lanes with >= MIN_LANE_ROWS
    answerable held-out rows; two-stage clustered bootstrap SE over those lanes."""
    counts = deltas.groupby("dataset").size()
    eligible = counts[counts >= MIN_LANE_ROWS].index
    voting = deltas[deltas["dataset"].isin(eligible)]
    point = float(voting.groupby("dataset")["delta"].mean().mean())
    se = pooled_bootstrap(voting, clusters)
    lo, hi = point - 1.96 * se, point + 1.96 * se
    state = "PASS" if lo > DELTA_MIN else ("FAIL" if hi < DELTA_MIN else "INCONCLUSIVE")
    return {"point": point, "se": se, "ci": (lo, hi), "state": state,
            "voting_lanes": len(eligible), "reported_only_lanes": int((counts < MIN_LANE_ROWS).sum())}


def show(label: str, r: dict) -> None:
    print(f"{label}: delta {r['point']:+.4f}, 95% CI [{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}], "
          f"SE {r['se']:.4f}\n  vs Δmin={DELTA_MIN} → {r['state']}   "
          f"(voting lanes {r['voting_lanes']}, reported-only {r['reported_only_lanes']})")

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1.2 + C1 — grouped split (clusters never straddle), router − per-lane train-selected
# constant, answerable held-out rows, amended pooling. THE claim measurement.
train_g, test_g = grouped_split_random(merged, clusters)
deltas_grouped = fit_and_deltas(train_g, test_g, baseline="per_lane")
c1 = pooled(deltas_grouped)
show("C1 (grouped split)", c1)

# leakage estimate: same measurement on the UNGROUPED split (the historical protocol,
# where duplicate clusters can straddle train/test). leakage = ungrouped − grouped.
train_u, test_u = exp.split("random_within_lane")
deltas_ungrouped = fit_and_deltas(train_u, test_u, baseline="per_lane")
c1_u = pooled(deltas_ungrouped)
show("same, ungrouped split", c1_u)
print(f"\nleakage estimate (ungrouped − grouped): {c1_u['point'] - c1['point']:+.4f}")

C1 (grouped split): delta -0.0154, 95% CI [-0.0297, -0.0010], SE 0.0073
  vs Δmin=0.02 → FAIL   (voting lanes 9, reported-only 32)


same, ungrouped split: delta -0.0227, 95% CI [-0.0484, +0.0030], SE 0.0131
  vs Δmin=0.02 → FAIL   (voting lanes 9, reported-only 30)

leakage estimate (ungrouped − grouped): -0.0074


In [3]:
# 1.3 — decisive vs answerable, side by side, same grouped split. The headline has
# historically been the decisive slice; a customer's workload is the answerable set.
score_cols = ["score_dense_only", "score_pure_rrf", "score_sparse_only"]
test_scores = merged.loc[deltas_grouped.index, score_cols].to_numpy()
margins = np.sort(test_scores, axis=1)
decisive_mask = (margins[:, -1] - margins[:, -2]) >= 0.4

rows = []
for label, sel in [("answerable (all)", np.ones(len(deltas_grouped), bool)),
                   ("decisive slice", decisive_mask),
                   ("non-decisive answerable", ~decisive_mask)]:
    sub = deltas_grouped[sel]
    rows.append({"slice": label, "rows": len(sub),
                 "row-weighted delta": float(sub["delta"].mean()),
                 "share of answerable": len(sub) / len(deltas_grouped)})
pd.DataFrame(rows).round(4)

,slice,rows,row-weighted delta,share of answerable
0,answerable (all),7597,-0.0196,1.0000
1,decisive slice,1015,0.0027,0.1336
2,non-decisive answerable,6582,-0.0231,0.8664


In [4]:
# 1.4 — the hide-one-lane rotation (all lanes), amended pooling, router − train-global
# constant. Supporting readout only (§8: lanes are a diversity instrument, not a transfer
# sample); the (grouped-random − rotation) gap is d44(c)'s corpus-dependence measurement.
lane_frames = []
for lane_name in sorted(merged["dataset"].unique()):
    d = fit_and_deltas(merged[merged["dataset"] != lane_name],
                       merged[merged["dataset"] == lane_name], baseline="global")
    if not d.empty:
        lane_frames.append(d)
deltas_rot = pd.concat(lane_frames)
rot = pooled(deltas_rot)
show("C3 rotation (supporting readout)", rot)
print(f"corpus-dependence gap (grouped-random − rotation): {c1['point'] - rot['point']:+.4f}")
deltas_rot.groupby("dataset")["delta"].agg(["mean", "count"]).sort_values("mean").round(3)

C3 rotation (supporting readout): delta +0.0234, 95% CI [-0.0101, +0.0568], SE 0.0170
  vs Δmin=0.02 → INCONCLUSIVE   (voting lanes 20, reported-only 21)
corpus-dependence gap (grouped-random − rotation): -0.0387


,mean,count
dataset,,
crumb-theorem-retrieval,-0.884,1
freshstack-godot,-0.174,62
bright-psychology,-0.121,63
bright-sustainable-living,-0.098,74
bright-biology,-0.094,79
bright-stackoverflow,-0.087,84
bright-robotics,-0.075,60
bright-earth-science,-0.066,98
freshstack-laravel,-0.063,150


In [5]:
# Probe gate — the served model (refit on ALL data, the artifact a customer gets),
# both frozen sets, ordering criterion. One hand-ruled failure = gate RED.
from hybrid_search_rrf_dataset.probes import probe_ordering

served = StrategyRouter(Representation.ENGINEERED, delta=0.05).fit(data, all_rows="all")
served.tune_thresholds(data)
table = probe_ordering(served)
decided = table[table["ordering_ok"].notna()]
fails = decided[~decided["ordering_ok"].astype(bool)]
print(f"probe gate: {len(decided) - len(fails)}/{len(decided)} decided cases pass the "
      f"ordering criterion → gate {'GREEN' if fails.empty else 'RED'}")
print(f"failures by set: {fails['set'].value_counts().to_dict()}")
table.round(3)

probe gate: 29/36 decided cases pass the ordering criterion → gate RED
failures by set: {'golden': 6, 'archetype': 1}


,set,query,expected,served,p_dense,p_sparse,ordering_ok
0,archetype,a3f5d8b9e12c4d56789abcdef0123456,sparse_only,dense_only,0.260,0.486,True
1,archetype,/etc/nginx/nginx.conf,sparse_only,sparse_only,0.475,0.543,True
2,archetype,ERR_CONNECTION_RESET,sparse_only,sparse_only,0.406,0.514,True
3,archetype,explain quicksort,dense_only,dense_only,0.410,0.488,False
4,archetype,HTTP 502,corpus-decides,dense_only,0.641,0.477,<NA>
5,archetype,comment volent les oiseaux,dense_only,dense_only,0.528,0.456,True
6,archetype,como aprender a programar en rust,dense_only,dense_only,0.528,0.430,True
7,golden,who founded apple?,dense_only,dense_only,0.591,0.380,True
8,golden,how does photosynthesis work in plants,dense_only,dense_only,0.516,0.373,True
9,golden,explain quicksort,dense_only,dense_only,0.410,0.488,False


In [6]:
# 1.5 — per-archetype diagnostic (no threshold reads this): where does the router help
# or hurt, by query surface. Signature = has identifier features x short/long.
id_cols = [c for c in merged.columns if c.startswith("structured_identifiers.")]
test_rows = merged.loc[deltas_grouped.index]
sig = pd.DataFrame({
    "has_identifier": (test_rows[id_cols].fillna(0).sum(axis=1) > 0),
    "short": test_rows["query"].str.split().str.len() <= 4,
})
diag = (deltas_grouped.assign(**sig)
        .groupby(["has_identifier", "short"])["delta"]
        .agg(["mean", "count"]).round(4))
diag

mean  count
has_identifier short               
False          False -0.0145   4714
               True  -0.0404    790
True           False -0.0197   1783
               True  -0.0448    310

In [7]:
# Summary — what goes into VERDICT.md
print("C1 (amended estimator, grouped split):", c1["state"],
      f"delta {c1['point']:+.4f} CI [{c1['ci'][0]:+.4f}, {c1['ci'][1]:+.4f}]")
print(f"leakage (ungrouped − grouped): {c1_u['point'] - c1['point']:+.4f}")
print("rotation readout:", rot["state"], f"delta {rot['point']:+.4f}")
print(f"probe gate: {'RED' if not fails.empty else 'GREEN'} ({len(fails)} ordering failures)")

C1 (amended estimator, grouped split): FAIL delta -0.0154 CI [-0.0297, -0.0010]
leakage (ungrouped − grouped): -0.0074
rotation readout: INCONCLUSIVE delta +0.0234
probe gate: RED (7 ordering failures)
